# Isaac Scene Generator Example Notebook

This is an example notebook, created to paraphrase the [scene_generator tool](https://gitlab-master.nvidia.com/amousavian/scene_generator) as described in the [requirements document](https://docs.google.com/document/d/1wHpOCh4fgrEssKBFdCRpZQlvxkCosVBMcIAJalnK9GA/edit?usp=sharing).

Note: This code started as a direct copy of [`sample_syntheticdata`](https://isaac.gitlab-master-pages.nvidia.com/omni_isaac_sim/app_isaacsim/app_isaacsim/sample_syntheticdata.html#basic-sample) and was iterated on.

## Setup

You will need to import `nest_asyncio` so that IsaacSim and jupyter notebooks will play nice together.  This first code block does that, as well as printing out the version of python you are using, check that it is located at `<isaacSimLocation>/kit/python/bin/python3`.

In [ ]:
import nest_asyncio
import sys

nest_asyncio.apply()
print(sys.executable)

## Create a Headless IsaacSim

You should only have to do this part once; it will load kit, or at least the headless IsaacSim version of kit, by calling
`headlessIsaacSim = OmniKitHelper(config=CONFIG)`.

This will take some time, so give it a minute and wait for it to say "Hi".

In [ ]:
import os
import getpass

import omni
from omni.isaac.python_app import OmniKitHelper

user = getpass.getuser()

CONFIG = {
    "headless": True,
    "renderer": "RayTracedLighting",
    "experience": f'{os.environ["EXP_PATH"]}/omni.isaac.sim.python.kit',
    "temp_jupyter_stage": f'omniverse://ov-isaac-dev/Users/{user}/temp_jupyter_stage.usd',
}

headlessIsaacSim = OmniKitHelper(config=CONFIG)
print("Hi")

## Iterative Development

Once `headlessIsaacSim` is up and running in your notebook, the iterative development process begins. The iterative development process consists of creating a scene, or stage in usd lingo, and then viewing and interacting with it. There are two ways to view and interact with your stage; with `headlessIsaacSim` in this python notebook, or with an external Omniverse viewer, like vanilla Kit or IsaacSim.

### Creating the scene

In [ ]:
import random
import numpy as np

TRANSLATION_RANGE = 300.0
ROTATION_RANGE = 180.0
SCALE = 50.0

# SCENE SETUP
# Add a distant light
headlessIsaacSim.create_prim("/World/Light", "DistantLight")
# Create 10 randomly positioned and coloured spheres and cube
# We will assign each a semantic label based on their shape (sphere/cube)
for i in range(10):
    prim_type = random.choice(["Cube", "Sphere"])
    prim = headlessIsaacSim.create_prim(
        f"/World/{prim_type}{i}",
        prim_type,
        rotation = (np.random.rand(3) * ROTATION_RANGE).tolist(),
        translation = (np.random.rand(3) * TRANSLATION_RANGE).tolist(),
        scale = (SCALE, SCALE, SCALE),
        attributes = {'primvars:displayColor': [np.random.rand(3).tolist()]},
        semantic_label = prim_type,
    )

headlessIsaacSim.update()

### Viewing the scene in the notebook

This next example does not change the scene (but it could if you used commands like the ones above), but it does use the IsaacSim's `SyntheticDataHelper` to view the data.  Specifically, it shows a color, depth, and segmentation view of the scene, and then displays them within the notebook.

In [ ]:
import copy
import matplotlib.pyplot as plt

from omni.isaac.synthetic_utils import SyntheticDataHelper
from omni.syntheticdata import visualize

sd_helper = SyntheticDataHelper()

# Get groundtruth
headlessIsaacSim.update()
viewport = omni.kit.viewport.get_default_viewport_window()

sensor_names = [
    "rgb",
    "depth",
    "semanticSegmentation",
]

# appear to need initialization first, even though it runs again in get_groundtruth by default.
sd_helper.initialize(viewport, sensor_names)
gt = sd_helper.get_groundtruth(sensor_names, viewport)

# GROUNDTRUTH VISUALIZATION

# Setup a figure
_, axes = plt.subplots(1, 3, figsize=(20, 7))
axes = axes.flat
for ax in axes:
    ax.axis("off")

# RGB
axes[0].set_title("RGB")
for ax in axes[:-1]:
    ax.imshow(gt["rgb"])

# DEPTH
axes[1].set_title("Depth")
depth_data = np.clip(gt["depth"], 0, 255)
axes[1].imshow(visualize.colorize_depth(depth_data.squeeze()))

# SEMANTIC SEGMENTATION
axes[2].set_title("Semantic Segmentation")
semantic_seg = gt["semanticSegmentation"]
semantic_rgb = visualize.colorize_segmentation(semantic_seg)
axes[2].imshow(semantic_rgb, alpha=0.7)


### Viewing the scene Omniverse

Because the scene you are working with is saved on a nucleus server, you can see it updated live with any Omniverse view.  So start kit and open the `temp_jupyter_stage` file that was named in `CONFIG` when `headlessIsaacSim` was created.

![How to turn on live update.](turn_on_live_update.jpg "Turn On Live Update")

Once the scene is open, make sure live update is on for the root layer of the stage by clicking the cloud in the layer window so that it turns <span style="color:green">*green*</span>.

## Useful Functions

In [ ]:
# You can reset the stage at any time
headlessIsaacSim.reset_stage()

In [ ]:
# You can kernel restart within the jupyter notebook, but if you only want to restart headlessIsaacSim and
# not your entire python session, then you have to shutdown the running session.
headlessIsaacSim.shutdown()